# 🔵 Colab E-iv — TensorFlow Sequential / High-Level API
**Sequential → compile → fit → evaluate → predict**

In [ ]:
import tensorflow as tf, numpy as np, matplotlib.pyplot as plt
tf.random.set_seed(42); np.random.seed(42)
print('TF:', tf.__version__)

## 📊 Section 1 — Data

In [ ]:
N=1000
x1=np.random.uniform(-2,2,(N,1)); x2=np.random.uniform(-2,2,(N,1)); x3=np.random.uniform(-2,2,(N,1))
y=2*x1**2+3*x2*x3+np.sin(x1*x2)+0.5*x3**2+np.random.normal(0,0.1,(N,1))
X=np.hstack([x1,x2,x3])
X_n=(X-X.mean(0))/X.std(0); y_mean,y_std=float(y.mean()),float(y.std()); y_n=(y-y_mean)/y_std
sp=int(0.8*N)
Xtr,Xte=X_n[:sp].astype('float32'),X_n[sp:].astype('float32')
Ytr,Yte=y_n[:sp].astype('float32'),y_n[sp:].astype('float32')

## 🏗️ Section 2 — Sequential Model

In [ ]:
# Sequential: pass a list of layers — most concise Keras style
model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu',  kernel_initializer='he_normal', input_shape=(3,)),
    tf.keras.layers.Dense(32, activation='relu',  kernel_initializer='he_normal'),
    tf.keras.layers.Dense(16, activation='tanh'),
    tf.keras.layers.Dense(1)
], name='sequential_regression')

model.summary()

## 🏋️ Section 3 — compile + fit + Callbacks

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='mse',
    metrics=['mae', 'mse']
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=60,
                                      restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                          patience=30, verbose=1),
    tf.keras.callbacks.ModelCheckpoint('best_sequential.h5', monitor='val_loss',
                                        save_best_only=True, verbose=0)
]

history = model.fit(
    Xtr, Ytr,
    epochs=1000, batch_size=64,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=1
)

## 📈 Section 4 — Comprehensive Results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss curves
axes[0].semilogy(history.history['loss'],label='Train',color='steelblue')
axes[0].semilogy(history.history['val_loss'],'r--',label='Val')
axes[0].set(title='Training Curves',xlabel='Epoch',ylabel='MSE'); axes[0].legend(); axes[0].grid(alpha=0.3)

# Predicted vs actual
yp = model.predict(Xte, verbose=0)*y_std+y_mean
yt = Yte*y_std+y_mean
axes[1].scatter(yt,yp,alpha=0.4,s=15,color='royalblue')
mn,mx=yt.min(),yt.max(); axes[1].plot([mn,mx],[mn,mx],'r--',lw=2)
axes[1].set(title='Predicted vs Actual',xlabel='Actual',ylabel='Predicted'); axes[1].grid(alpha=0.3)

# Residuals
residuals = (yp - yt).flatten()
axes[2].hist(residuals,bins=30,color='royalblue',edgecolor='white',alpha=0.8)
axes[2].axvline(0,color='red',lw=2,linestyle='--')
axes[2].set(title='Residuals Distribution',xlabel='Prediction Error',ylabel='Count')
axes[2].grid(alpha=0.3)

plt.suptitle('Colab E-iv — TF Sequential', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

results = model.evaluate(Xte, Yte, verbose=0)
print(f'Test MSE: {results[0]:.4f} | MAE: {results[1]:.4f}')

## 🔁 Section 5 — Save, Reload, and All-Colab Comparison

In [ ]:
# Save full model
model.save('colab_e4_model')
reloaded = tf.keras.models.load_model('colab_e4_model')
yp2 = reloaded.predict(Xte[:5], verbose=0)
print('Reload diff:', np.abs(yp2 - model.predict(Xte[:5], verbose=0)).max())
print('Reloaded model produces identical predictions ✅')